In [1]:
import warnings
warnings.filterwarnings("ignore")

import xarray as xr
import uxarray as ux
import numpy as np
from datetime import datetime
from datetime import timedelta

In [2]:
day = "20260610"
TODAY = f"{day}12"
hour = 3
initTime = datetime.strptime(TODAY,"%Y%m%d%H")
time_fmt = "%Y-%m-%d %H:%M"

minLon, maxLon = -104.1, -80
minLat, maxLat = 32.65, 48.6

In [3]:
hrrr_obs = f"/glade/work/jaye/valpo/hrrr/"
mrms_obs = f"/glade/work/jaye/valpo/mrms/"
mpas_20_dir = f"/glade/derecho/scratch/jaye/valpo/mpas_runs/{day}/rundir_20km/"
grid_20 = f"{mpas_20_dir}x1.11981.grid.valpo.nc"
mpas_20_files = f"{mpas_20_dir}diag*"

In [4]:
uxds_20 = ux.open_mfdataset(grid_20,mpas_20_files,combine="by_coords")
times = len(uxds_20["Time"])*hour

In [9]:
print("--> Initializing NetCDF compilation pipeline...")
hourly_datasets = []

for fhr in range(0, times, hour):
    validTime = initTime + timedelta(hours=(float(fhr)))
    timestamp = validTime.strftime('%Y%m%d%H')
    obs_file = f"{hrrr_obs}hrrr_surface_obs_{timestamp}.grib2"
    
    ds_t2m = xr.open_dataset(obs_file, engine="cfgrib", backend_kwargs={"filter_by_keys": {"typeOfLevel": "heightAboveGround", "level": 2, "stepType": "instant"}, "indexpath": ""})
    ds_10m = xr.open_dataset(obs_file, engine="cfgrib", backend_kwargs={"filter_by_keys": {"typeOfLevel": "heightAboveGround", "level": 10, "stepType": "instant"}, "indexpath": ""})
    ds_mslp = xr.open_dataset(obs_file, engine="cfgrib", backend_kwargs={"filter_by_keys": {"shortName": "mslma"}, "indexpath": ""})
    ds_cape = xr.open_dataset(obs_file, engine="cfgrib", backend_kwargs={"filter_by_keys": {"shortName": "cape", "typeOfLevel": "surface"}, "indexpath": ""})
    
    lon_shifted = ((ds_t2m.longitude + 180) % 360) - 180
    
    hourly_ds = xr.Dataset(data_vars={"t2m": (["y","x"],ds_t2m["t2m"].values),"u10": (["y","x"],ds_10m["u10"].values),"v10": (["y","x"],ds_10m["v10"].values),
                                      "mslp": (["y","x"],ds_mslp["mslma"].values),"cape": (["y", "x"], ds_cape["cape"].values)},
                           coords={"longitude": (["y","x"],lon_shifted.values),"latitude": (["y","x"],ds_t2m.latitude.values),"time": validTime})
    
    spatial_mask = ((hourly_ds.longitude >= minLon - 1) & (hourly_ds.longitude <= maxLon + 1) & 
                    (hourly_ds.latitude >= minLat - 1) & (hourly_ds.latitude <= maxLat + 1))
    hourly_cropped = hourly_ds.where(spatial_mask, drop=True)
    
    hourly_datasets.append(hourly_cropped)

compiled_obs = xr.concat(hourly_datasets, dim="time")
compiled_obs.to_netcdf(f"{hrrr_obs}hrrr_surface_obs_{day}.nc")
print("--> Processed NetCDF file generated and saved successfully!")

--> Initializing NetCDF compilation pipeline...
--> Processed NetCDF file generated and saved successfully!


In [12]:
hourly_mrms = []

for fhr in range(0, times, hour):
    validTime = initTime + timedelta(hours=float(fhr))
    
    rain_file = f"{mrms_obs}mrms_rain_obs_{validTime.strftime('%Y%m%d%H')}.grib2"
    refl_file = f"{mrms_obs}mrms_refl_obs_{validTime.strftime('%Y%m%d%H')}.grib2"
        
    ds_rain = xr.open_dataset(rain_file, engine="cfgrib", backend_kwargs={"indexpath": ""})
    rain_slice = ds_rain["unknown"].sum(dim="time", skipna=True)
    
    if os.path.exists(refl_file) and os.path.getsize(refl_file) > 0:
        ds_refl = xr.open_dataset(refl_file, engine="cfgrib", backend_kwargs={"indexpath": ""})
        refl_slice = ds_refl["unknown"].squeeze().values
    else:
        print(f"  --> Warning: Using blank canvas for missing radar hour {fhr}")
        refl_slice = rain_slice.values * 0.0
    
    ds_hour = xr.Dataset(data_vars={"tp": (["latitude", "longitude"], rain_slice.values),"dbz": (["latitude", "longitude"], refl_slice)},
                         coords={"latitude": rain_slice.latitude,"longitude": rain_slice.longitude,"time": validTime}).expand_dims("time")
    
    ds_hour = ds_hour.assign_coords(longitude=((ds_hour.longitude + 180) % 360) - 180)
    
    mask = (ds_hour.longitude >= minLon - 1) & (ds_hour.longitude <= maxLon + 1) & \
           (ds_hour.latitude >= minLat - 1) & (ds_hour.latitude <= maxLat + 1)
    
    hourly_mrms.append(ds_hour.where(mask, drop=True).compute())

mrms_master = xr.concat(hourly_mrms, dim="time")
mrms_master = mrms_master.rename({"latitude": "lat_mrms","longitude": "lon_mrms"})
mrms_master.to_netcdf(f"{mrms_obs}mrms_obs_{day}.nc")
print("--> Done! Master NetCDF observation file compiled with no collisions.")

--> Done! Master NetCDF observation file compiled with no collisions.
